In [1]:
# mol,ctag,basis,total_energy,optical_lumo,gap,homo,lumo,spectral_overlap,delta_homo,delta_lumo,delta_optical_lumo,homo_extrapolated 12,lumo_extrapolated 13,gap_extrapolated 14,optical_lumo_extrapolated,smile 16
# "blank
#  OpenBabel03101821053D

In [3]:
from sklearn.metrics import mean_absolute_error
from torch_geometric.data import Data
import torch
from rdkit.Chem import AllChem
from rdkit.Chem import MACCSkeys
import numpy as np
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Fragments

In [4]:
filename = r'C:\Users\AI1-PC\PycharmProjects\main\datasets\NREL_opv\NREL_opv_dataset_DFT_geom.scv'

xyz_list_dft = []
atomic_number_list_dft = []
optical_lumo_list_dft = []
gap_list_dft = []
homo_list_dft = []
lumo_list_dft = []
spectral_overlap_list_dft = []
homo_extr_list_dft = []
lumo_extr_list_dft = []
gap_extr_list_dft = []
smiles_list_dft = []

data_list_dft = []

with open(filename) as file:
    for line in file:
        rline = line.rstrip()
        
        if len(rline.split()) == 16:
            xyz_atom_string = rline.split()
            
            x = float(xyz_atom_string[0])
            y = float(xyz_atom_string[1])
            z = float(xyz_atom_string[2])
            
            xyz = [x, y, z]
            
            atom_symbol = xyz_atom_string[3]
            atomic_number = Chem.GetPeriodicTable().GetAtomicNumber(atom_symbol)
            
            xyz_list_dft.append(xyz)
            atomic_number_list_dft.append(atomic_number)
            
        if 'b3lyp/6-31g(d)' in rline:
            meas_string = rline.split(',')

            gap_list_dft.append(float(meas_string[5]))
            homo_list_dft.append(float(meas_string[6]) * -1)
            lumo_list_dft.append(float(meas_string[7]) * -1)
            spectral_overlap_list_dft.append(float(meas_string[8]))
            
            # homo_extr_list.append(float(meas_string[12]))
            # lumo_extr_list.append(float(meas_string[13]))
            # gap_extr_list.append(float(meas_string[14]))
            smiles_list_dft.append(meas_string[16])
            
        if 'camb3lyp/6-31g' in rline:
            meas_string = rline.split(',')

            gap_list_dft.append(float('nan'))
            homo_list_dft.append(float('nan'))
            lumo_list_dft.append(float('nan'))
            spectral_overlap_list_dft.append(float('nan'))

        if 'OpenBabel' in rline:
            x = torch.tensor(atomic_number_list_dft, dtype=torch.long)
            pos = torch.tensor(xyz_list_dft, dtype=torch.float32)
            y = torch.tensor([[homo_list_dft[-1], lumo_list_dft[-1]]], dtype=torch.float32)

            data = Data(
                x=x,
                pos=pos,
                y=y
            )

            data_list_dft.append(data)

            xyz_list_dft = []
            atomic_number_list_dft = []

            print(len(data_list_dft))

IndexError: list index out of range

In [ ]:
from collections import Counter

smiles_counter = Counter(smiles_list_dft)
duplicates = {smiles: count for smiles, count in smiles_counter.items() if count > 1}
print(f"Найдено дубликатов: {len(duplicates)}")

if duplicates:
    print("Примеры дубликатов:")
    for smiles, count in list(duplicates.items()):
        print(f"  {smiles}: {count} раз")

In [3]:
def has_nan_in_data(data):
    for key, value in data:
        if torch.is_tensor(value):
            if torch.isnan(value).any():
                return True
    return False

filtered_data_list_dft = [data for data in data_list_dft if not has_nan_in_data(data)]

In [4]:
def remove_duplicates_by_smiles(data_list, smiles_list):
    seen_smiles = set()
    unique_data_list = []
    unique_smiles_list = []
    unique_indices = []
    
    for i, (data, smiles) in enumerate(zip(data_list, smiles_list)):
        if smiles not in seen_smiles:
            seen_smiles.add(smiles)
            unique_data_list.append(data)
            unique_smiles_list.append(smiles)
            unique_indices.append(i)
    
    print(f"Было: {len(data_list)}, стало: {len(unique_data_list)}")
    print(f"Удалено дубликатов: {len(data_list) - len(unique_data_list)}")
    
    return unique_data_list, unique_smiles_list, unique_indices

filtered_unique_data_list_dft, unique_smiles_list_dft, unique_indices_dft = remove_duplicates_by_smiles(filtered_data_list_dft, smiles_list_dft)

Было: 94576, стало: 91088
Удалено дубликатов: 3488


In [6]:
len(filtered_unique_data_list_dft)

91088

In [60]:
filtered_unique_data_list_dft[0].x

tensor([ 6,  6,  6,  6,  6,  6,  6,  6,  6,  6,  6,  6,  6,  6,  1,  8,  8,  8,
         8,  7,  7,  6,  9,  9,  9,  6,  9,  9,  9,  6,  1,  1,  1,  6,  1,  1,
         1,  6,  6,  6,  6,  6,  1, 16,  7,  7,  6,  1,  1,  1,  6,  1,  1,  1])

In [7]:
data_raw = pd.read_csv(r'C:\Users\AI1-PC\PycharmProjects\main\datasets\polymers\DOI 10.1021acs.jpclett.8b00635_exp.txt', sep=',', index_col=None)
data_no_dub = data_raw.drop_duplicates(subset=['SMILES'])

In [8]:
homo_list_exp = data_no_dub['-HOMO (eV)'].to_numpy()
lumo_list_exp = data_no_dub['-LUMO (eV)'].to_numpy()
gap_list_exp = data_no_dub['bandgap(eV)'].to_numpy()
smiles_list_exp = data_no_dub['SMILES'].to_numpy()

In [9]:
len(smiles_list_exp)

1163

In [10]:
dft_thio_mask = []
for i in range(len(unique_smiles_list_dft)):
    m = Chem.MolFromSmiles(unique_smiles_list_dft[i])
    if Chem.Fragments.fr_thiophene(m) != 0:
        dft_thio_mask.append(i)

In [11]:
unique_smiles_list_dft_thio = np.array(unique_smiles_list_dft)[dft_thio_mask]

In [ ]:
exp_thio_mask = []
smiles_list_exp_thio = []
for i in range(len(smiles_list_exp)):
    m = Chem.MolFromSmiles(smiles_list_exp[i])
    if Chem.Fragments.fr_thiophene(m) != 0:
        exp_thio_mask.append(i)

In [44]:
smiles_list_exp_thio = np.array(smiles_list_exp)[exp_thio_mask]
homo_list_exp_thio = homo_list_exp[exp_thio_mask]
lumo_list_exp_thio = homo_list_exp[exp_thio_mask]

In [ ]:
def get_molecular_similarity(dataset1_smiles, dataset2_smiles, top_k=10, fingerprint_type='morgan'):

    mols1 = [Chem.MolFromSmiles(smi) for smi in dataset1_smiles]
    mols2 = [Chem.MolFromSmiles(smi) for smi in dataset2_smiles]

    mols1 = [mol for mol in mols1 if mol is not None]
    mols2 = [mol for mol in mols2 if mol is not None]

    if fingerprint_type == 'morgan':
        fps1 = [AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048) for mol in mols1]
        fps2 = [AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048) for mol in mols2]
    elif fingerprint_type == 'maccs':
        fps1 = [MACCSkeys.GenMACCSKeys(mol) for mol in mols1]
        fps2 = [MACCSkeys.GenMACCSKeys(mol) for mol in mols2]

    fp_array1 = np.array([list(fp) for fp in fps1])
    fp_array2 = np.array([list(fp) for fp in fps2])

    nbrs = NearestNeighbors(n_neighbors=top_k, metric='jaccard', n_jobs=-1)
    nbrs.fit(fp_array1)

    distances, indices = nbrs.kneighbors(fp_array2)
    
    return indices, distances

dataset1_smiles = unique_smiles_list_dft_thio
dataset2_smiles = smiles_list_exp_thio

similar_indices, similarity_distances = get_molecular_similarity(
    dataset1_smiles, dataset2_smiles, top_k=10
)

unique_similar_molecules = set()
for row in similar_indices:
    unique_similar_molecules.update(row)

similar_molecules_from_dataset1 = [dataset1_smiles[i] for i in unique_similar_molecules]

In [16]:
len(similar_molecules_from_dataset1)

2660

In [19]:
similar_mask = []
for i in range(len(unique_smiles_list_dft)):
    if unique_smiles_list_dft[i] in similar_molecules_from_dataset1:
        similar_mask.append(i)

In [24]:
data_list_dft_pretrain = np.array(filtered_unique_data_list_dft, dtype=object)[dft_thio_mask]
data_list_dft_pretrain_focus = np.array(filtered_unique_data_list_dft, dtype=object)[similar_mask]

In [25]:
len(data_list_dft_pretrain_focus)

2660

In [ ]:
from rdkit.Chem import rdDepictor, AddHs

mols_list_exp_thio = []

for s in smiles_list_exp_thio:
    mol = Chem.AddHs(Chem.MolFromSmiles(s))
    molH = AddHs(mol)
    rdDepictor.Compute2DCoords(molH)
    
    mols_list_exp_thio.append(molH)

In [83]:
from numpy.random._examples.cffi.extending import rng
import torchani
import torch
import numpy as np

def optimize_molecule_ani(mol, max_steps=500, convergence=1e-4):
    
    atoms = []
    coordinates = []
    
    for atom in mol.GetAtoms():
        atoms.append(atom.GetAtomicNum())
    
    for i in range(mol.GetNumAtoms()):
        pos = mol.GetConformer().GetAtomPosition(i)
        new_z = pos.z + rng.uniform(-0.1, 0.1)
        coordinates.append([pos.x, pos.y, new_z])
    
    model = torchani.models.ANI2x(periodic_table_index=True).to('cuda')
    
    atoms = torch.tensor([atoms]).to('cuda')
    coordinates = torch.tensor([coordinates], dtype=torch.float32).to('cuda').requires_grad_(True)

    optimizer = torch.optim.LBFGS([coordinates], 
                                 lr=0.1, 
                                 max_iter=20, 
                                 tolerance_grad=convergence,
                                 tolerance_change=convergence)
    
    energies = []
    
    def closure():
        optimizer.zero_grad()
        energy = model((atoms, coordinates)).energies
        energy.backward()
        return energy
    
    for step in range(max_steps):
        current_energy = optimizer.step(closure).item()
        energies.append(current_energy)

        if step > 10 and abs(energies[-1] - energies[-2]) < convergence:
            print(f"Сходимость достигнута на шаге {step}")
            break
            
        if step % 50 == 0:
            print(f"Шаг {step}: Энергия = {current_energy:.6f} kcal/mol")
    
    return atoms.cpu().reshape(-1), coordinates.detach().cpu().reshape(-1,3)

# water_species = torch.tensor([[8, 1, 1]])  
# water_coords = torch.tensor([[[0.0, 0.0, 0.0],      
#                              [1.5, 1.5, 0.0],       
#                              [-1.5, 1.5, 0.0]]],    
#                            dtype=torch.float32)

atoms, optimized_coords = optimize_molecule_ani(mols_list_exp_thio[1])

C:/Users/MSI/PycharmProjects/pythonProject/.venv/Lib/site-packages/torchani\resources/
Шаг 0: Энергия = -3093.926714 kcal/mol
Шаг 50: Энергия = -3785.169603 kcal/mol
Сходимость достигнута на шаге 61


In [84]:
optimized_coords

tensor([[ 8.5343e+00,  3.6736e+00,  3.3314e-01],
        [ 4.3467e+00,  3.4258e+00,  3.4079e-02],
        [ 4.3423e+00,  3.5980e+00, -1.4557e-01],
        [ 4.4400e+00,  3.3313e+00,  4.0752e-02],
        [ 2.6523e+00,  1.7230e+00, -2.5833e-02],
        [ 3.0966e+00, -7.0291e-01, -3.3989e-01],
        [ 2.2371e+00, -3.7913e+00,  1.2473e-01],
        [ 3.6250e+00, -5.8622e+00,  3.3265e-02],
        [ 2.7165e+00, -4.9198e+00,  7.6153e-01],
        [-1.0248e+00, -7.5560e+00, -7.5686e-01],
        [-6.0258e+00, -7.2684e+00,  6.5669e-01],
        [-1.1131e+01, -4.8559e+00,  1.1142e+00],
        [-7.5851e+00,  1.5368e+00,  1.7301e-01],
        [-4.3130e+00,  2.2124e+00, -7.2393e-01],
        [ 5.1825e+00, -3.4257e+00,  2.8217e-02],
        [ 5.2897e+00, -4.6678e+00, -5.6992e-02],
        [ 9.3454e+00, -2.7393e+00,  1.3794e-01],
        [ 1.0077e+01, -8.0961e-01, -5.6986e-02],
        [ 1.0291e+01, -3.6052e-01, -5.7888e-02],
        [ 1.2032e+01,  2.0958e-02, -5.4582e-01],
        [ 1.3167e+01

In [86]:
mmm = mols_list_exp_thio[1]

In [91]:
from rdkit.Chem.rdForceFieldHelpers import MMFFOptimizeMolecule

for mol in mols_list_exp_thio:
    AllChem.EmbedMolecule(mol, maxAttempts=100, useRandomCoords=True)
    MMFFOptimizeMolecule(mol, maxIters=10000, mmffVariant="MMFF94s")


[19:42:07] UFFTYPER: Unrecognized atom type: Se2+2 (30)


KeyboardInterrupt: 

In [4]:
# for i in range(len(filtered_data_list_dft)):
#     torch.save(filtered_data_list_dft[i],
#                fr'C:\Users\MSI\PycharmProjects\pythonProject\NIOC\polymer_project\datasets\NREL_opv\NREL_opv_dataset_DFT_geom_to_prop_dimenet\{i}.pt')

NameError: name 'filtered_data_list' is not defined